# Score Caldova feedback with Fabric AI Functions

This notebook loads synthetic Caldova product and feedback data, adds product context to each comment, applies `ai.analyze_sentiment`, validates the labels, and writes managed Delta tables for a Fabric data agent or ontology.

In [ ]:
import synapse.ml.spark.aifunc as aifunc
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, IntegerType, StringType, StructField, StructType

DATA_ROOT = "/lakehouse/default/Files/caldova-sentiment"
SUPPORTED_SENTIMENTS = ["positive", "negative", "neutral", "mixed"]

In [ ]:
product_schema = StructType([
    StructField("ProductID", StringType(), False),
    StructField("ProductName", StringType(), False),
    StructField("ProductDescription", StringType(), False),
])

feedback_schema = StructType([
    StructField("FeedbackID", IntegerType(), False),
    StructField("ProductID", StringType(), False),
    StructField("FeedbackDate", DateType(), False),
    StructField("Channel", StringType(), False),
    StructField("Region", StringType(), False),
    StructField("FeedbackText", StringType(), False),
])

product = (spark.read
           .option("header", True)
           .schema(product_schema)
           .csv(f"{DATA_ROOT}/products.csv"))

feedback = (spark.read
            .option("header", True)
            .option("dateFormat", "yyyy-MM-dd")
            .schema(feedback_schema)
            .csv(f"{DATA_ROOT}/customer_feedback.csv"))

if product.count() != 3:
    raise RuntimeError("Expected 3 product rows.")
if feedback.count() != 16:
    raise RuntimeError("Expected 16 feedback rows.")
if product.select("ProductID").distinct().count() != 3:
    raise RuntimeError("ProductID values must be unique.")
if feedback.select("FeedbackID").distinct().count() != 16:
    raise RuntimeError("FeedbackID values must be unique.")

orphaned_feedback = feedback.join(product, "ProductID", "left_anti").count()
if orphaned_feedback:
    raise RuntimeError(f"Found {orphaned_feedback} feedback rows without a product.")

In [ ]:
analysis_input = (feedback
                  .join(product, "ProductID", "inner")
                  .withColumn(
                      "AnalysisText",
                      F.concat_ws(
                          " ",
                          F.concat(F.lit("Product name: "), F.col("ProductName"), F.lit(".")),
                          F.concat(F.lit("Product context: "), F.col("ProductDescription")),
                          F.concat(F.lit("Customer feedback: "), F.col("FeedbackText")),
                      ),
                  ))

# This code uses AI. Always review output for mistakes.
scored = analysis_input.ai.analyze_sentiment(
    input_col="AnalysisText",
    output_col="Sentiment",
    error_col="AIError",
).withColumn("Sentiment", F.lower(F.col("Sentiment")))

In [ ]:
failed_rows = scored.filter(
    F.col("Sentiment").isNull()
    | (~F.col("Sentiment").isin(SUPPORTED_SENTIMENTS))
    | F.col("AIError").isNotNull()
)

if failed_rows.count():
    display(failed_rows.select("FeedbackID", "Sentiment", "AIError"))
    raise RuntimeError("One or more feedback records were not scored.")

product.write.format("delta").mode("overwrite").saveAsTable("product")

output_columns = [
    "FeedbackID", "ProductID", "ProductName", "ProductDescription",
    "FeedbackDate", "Channel", "Region", "FeedbackText",
    "AnalysisText", "Sentiment", "AIError",
]
(scored.select(*output_columns)
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("customer_feedback_sentiment"))

print("Validation passed: 3 products and 16 feedback records; all records have supported sentiment labels.")

In [ ]:
display(
    scored.groupBy("Sentiment")
          .count()
          .withColumnRenamed("count", "FeedbackCount")
          .orderBy("Sentiment")
)

display(
    scored.select(
        "FeedbackID", "ProductName", "Region", "FeedbackText", "Sentiment"
    ).orderBy("FeedbackID")
)